# W03-A · Guess the next word

Type *"i like"* on your phone and the keyboard offers three words. How does it know? It has read a lot of text, and it
remembers **what usually came next**.

A program that gives every possible next word a probability is called a **language model**. Today you build one from
12 short sentences, using nothing but counting. Real keyboards do the same with billions of sentences.

By the end of this notebook you will be able to:
1. mark where sentences start and end
2. count single words, and see why they make a poor guesser
3. count **pairs** of neighbouring words, and suggest the next word from them
4. turn the counts into **probabilities**
5. work out the probability of a whole sentence, and see why some get exactly **0**
6. let the model write its own sentences

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ads6703-nlp/blob/main/w03/W03-A-guess-the-next-word.ipynb) · On Colab, run the next cell first. On your laptop, you can skip it.

In [ ]:
# Only for Google Colab: fetch the course folder (with the data) and move into this week's folder.
# On your laptop this cell does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if os.path.exists("/content/ADS6703"):
        subprocess.run(["git", "-C", "/content/ADS6703", "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/shripada/ads6703-nlp.git", "/content/ADS6703"], check=True)
    os.chdir("/content/ADS6703/w03")

In [ ]:
# ▶ Setup: just run this cell. These 12 sentences are all the text our model will ever read.
from collections import Counter   # counting (see W03-0)
import random                     # random picks, for section 6

sentences = [
    "i like tea",
    "i like coffee",
    "you like tea",
    "i drink tea",
    "i like hot tea",
    "you drink coffee",
    "we like coffee",
    "we drink hot coffee",
    "i like cold coffee",
    "you like hot tea",
    "we like tea",
    "i drink coffee",
]
print(len(sentences), "sentences")

## 1 · Where does a sentence start and end?

A guesser needs to know two extra things: which words tend to **start** a sentence, and when a sentence is **finished**.
We add two made-up words to every sentence: `<s>` (start) at the front and `</s>` (end) at the back.

### ✍️ Exercise 1: `add_marks(sentence)`

Return a list: `"<s>"`, then the words of the sentence (`sentence.split()`), then `"</s>"`.

In [ ]:
def add_marks(sentence):
    return ...   # ✍️ hint: join three lists with +
print(add_marks("i like tea"))
assert add_marks("i like tea") == ["<s>", "i", "like", "tea", "</s>"], "not quite"
print("✅")

## 2 · The simplest guesser: the most common word

Suppose the keyboard ignored what you typed and just counted how often each word appears. A model that looks at one
word at a time, with no context, is called a **unigram** model (*uni* = one).

In [ ]:
word_counts = Counter()
for s in sentences:
    word_counts.update(add_marks(s)[1:])   # every word, plus </s>; skip <s>, which is never a guess
total = sum(word_counts.values())

print(word_counts.most_common())
print("P(like) =", word_counts["like"], "/", total)

> 📖 **Reading the output.** The most common "word" is the end mark `</s>` (every sentence has one), then *like*.
> The **probability** of a word is its count divided by the total: *like* is 8 out of 52 words.
>
> As a keyboard, this model is useless: whatever you type, it suggests the same words. To guess well, it must look at
> the word you just typed.

## 3 · Look at the word before: counting pairs

A pair of neighbouring words is called a **bigram** (*bi* = two). For every word, we count which words came right
after it. You did this in W03-0 for four sentences; now do it for all twelve.

### ✍️ Exercise 2: count every pair

Fill `pair_counts`, a `Counter` of `(word, next word)` pairs, from every sentence.
Hint: for each sentence, `marked = add_marks(s)`, then `zip(marked, marked[1:])` gives its pairs.

In [ ]:
pair_counts = Counter()
for s in sentences:
    marked = add_marks(s)
    ...   # ✍️ add the pairs of `marked` to pair_counts (hint: .update and zip)
for pair, n in pair_counts.most_common():
    print(n, pair)
assert pair_counts[("i", "like")] == 4, "not quite: did you count the pairs of every sentence?"
print("✅")

### ✍️ Exercise 3: `next_counts(word)`

Return a dict: each word seen right after `word` → how many times (the same function as W03-0, Exercise 4).

In [ ]:
def next_counts(word):
    result = {}
    for (w1, w2), n in pair_counts.items():
        ...   # ✍️ if the first word is `word`, store the count n under w2
    return result
print(next_counts("i"))
print(next_counts("like"))
assert next_counts("like") == {"tea": 3, "coffee": 2, "hot": 2, "cold": 1}, "not quite"
print("✅")

Now a keyboard is easy: suggest the words seen most often after the last word typed. `suggest` is written for you:
it sorts the next words by their count, biggest first, and keeps the first three.

In [ ]:
def suggest(word):
    after = next_counts(word)
    return sorted(after, key=after.get, reverse=True)[:3]   # biggest count first, keep 3

for w in ["<s>", "i", "like", "drink", "hot", "milk"]:
    print(f"{w:>6} → {suggest(w)}")

> 📖 **Reading the output.** After *like* the keyboard suggests *tea, coffee, hot*; after *hot*, *tea* and *coffee*.
> After `<s>` you see how sentences begin: mostly *i*. And after *milk*, a word it has never read, it has nothing to
> suggest at all.

## 4 · From counts to probabilities

After *"like"* we saw *tea* 3 times, out of 8 words in all. So, after *"like"*, the probability of *"tea"* is
3 / 8. We write it as **P(tea | like)**, read *"the probability of tea, given like"*:

$$P(\text{next} \mid \text{word}) = \frac{\text{how often the pair } \textit{word next} \text{ appears}}{\text{how often } \textit{word} \text{ is followed by anything}}$$

### ✍️ Exercise 4: `p_next(word, nxt)`

Return the probability that `nxt` comes right after `word`. If `word` was never seen, return `0`.
Hint: `after.get(nxt, 0)` gives the count of `nxt`, or 0 if it isn't there.

In [ ]:
def p_next(word, nxt):
    after = next_counts(word)
    if len(after) == 0:
        return 0
    return ...   # ✍️ the count of nxt (0 if missing), divided by the total of after
print("P(tea | like) =", p_next("like", "tea"))
print("P(i | <s>) =", p_next("<s>", "i"))
print("P(cold | drink) =", p_next("drink", "cold"))
assert p_next("like", "tea") == 3 / 8 and p_next("milk", "a") == 0, "not quite"
print("✅")

## 5 · The probability of a whole sentence

A sentence is a chain of guesses. The model guesses the first word after `<s>`, then the second after the first, and
so on until `</s>`. The probability of the whole sentence is all the steps **multiplied** together.

For *"i like hot tea"*:

P(i \| &lt;s&gt;) × P(like \| i) × P(hot \| like) × P(tea \| hot) × P(&lt;/s&gt; \| tea)

It is the same rule as tossing coins: the chance of heads *and then* heads again is ½ × ½.

### ✍️ Exercise 5: `sentence_prob(sentence)`

Add the marks, then multiply `p_next` over every pair.

In [ ]:
def sentence_prob(sentence):
    words = add_marks(sentence)
    p = 1
    for w1, w2 in zip(words, words[1:]):
        ...   # ✍️ multiply p by the probability of this step
    return p
assert abs(sentence_prob("i like hot tea") - 1 / 18) < 1e-12, "not quite"
print("✅")
for s in ["i like hot tea", "we like hot coffee", "tea hot like i", "i drink cold tea"]:
    print(f"{sentence_prob(s):.4f}   {s}")

> 📖 **Reading the output.**
> - *"i like hot tea"* is one of our 12 sentences, and gets about 0.056 (1/18).
> - *"we like hot coffee"* is **not** one of the 12, but every pair in it (*we like*, *like hot*, *hot coffee*) has been
>   seen, so it gets a probability above 0. The model can score sentences it has never read.
> - *"tea hot like i"* is word salad and gets **0**. Good.
> - *"i drink cold tea"* is normal English, yet it also gets **0**. Let's see why:

In [ ]:
words = add_marks("i drink cold tea")
for w1, w2 in zip(words, words[1:]):
    print(f"P({w2} | {w1}) = {p_next(w1, w2):.2f}")

> 📖 **Reading the output.** In our 12 sentences nobody wrote *drink cold*, so P(cold \| drink) = 0, and one zero
> step makes the whole product 0. The model believes *"i drink cold tea"* is impossible.
>
> This is the big weakness of counting: **never seen is not the same as impossible**.

## 6 · Let the model write sentences

A language model can also **generate** text: start at `<s>`, pick a next word **at random, in proportion to the
counts** (W03-0, section 6), then a next word after that, until it picks `</s>`.

In [ ]:
# ▶ Just run this cell, a few times.
def generate():
    word = "<s>"
    out = []
    while True:
        after = next_counts(word)
        word = random.choices(list(after), weights=list(after.values()))[0]   # a weighted random pick
        if word == "</s>":
            return " ".join(out)       # glue the words back together with spaces
        out.append(word)

random.seed(3)
for _ in range(8):
    s = generate()
    print(f"{s:<22} {'(one of the 12)' if s in sentences else '(NEW)'}")

> 📖 **Reading the output.** Some sentences are copies of the 12, but some are **new**: each of their pairs was seen
> somewhere, and the model glues them together. That is how a language model "writes": one guessed word at a time.
>
> Our model only looks **one word back**. Looking two words back (a **trigram** model, *tri* = three) would guess better,
> but needs far more text, because there are far more three-word combinations to count.

## Recap

| idea | what you saw |
|---|---|
| Language model | gives every possible next word a probability |
| `<s>` and `</s>` | let the model guess the first word, and decide when to stop |
| Unigram (one word) | ignores what you typed: always the same guesses |
| Bigram (pairs) | looks one word back: *like → tea, coffee, hot* |
| P(next \| word) | count of the pair ÷ count of the word |
| Sentence probability | multiply every step; one unseen pair makes it **0** |
| Generating | pick next words at random by their counts; makes new sentences |